In [1]:
using Pkg
using Metal

cd(@__DIR__)
Pkg.activate("../")
ParamFile = "../config/testparam.csv"  # maybe GeoPoints and planet1D should be fusioned

# batchGPU should be at this level (I have not made it as a module yet, since the choice of Metal/CUDA should be done in a manual way)
include("../src/batchFiles/batchGPU.jl")


include("../src/commonBatchs.jl")
include("../src/planet1D.jl")
include("../src/GeoPoints.jl")

include("../src/flexOPT.jl")

using .commonBatchs, .planet1D, .GeoPoints, .flexOPT

  Activating 

devs = Metal.devices() = Metal.MTL.MTLDeviceInstance[Metal.MTL.MTLDeviceInstance (object of type AGXG13XDevice)]

project at `~/Documents/Github/flexOPT`



→ Using Metal backend (1 device(s))
Selected backend type: MetalBackend


# model construction

In [2]:
using Symbolics,CairoMakie
numPointsX = collect(2:2)
logsOfHinverse = [1.0*i for i in 0:4]

cases=[]
#prefix="B"*string(tmpOrderBspace)*"_"*"w"*string(tmpWorderBspace)*"_"*string(tmpSupplementaryOrder)*"_"
prefix=""
L = 10.0*π
cases = push!(cases,(name=prefix*"λ_2",T=cos(x),β= 1))

@variables x
∂ = Differential(x)


misfit = Array{Float64,3}(undef,length(logsOfHinverse),length(cases),length(numPointsX))

modelFamily=[]

fig = Figure()
ax = Axis(fig[1, 1], xlabel="x", ylabel="model")
iH=5
for iCase ∈ eachindex(cases)
    name,_,β = cases[iCase]
    ΔxTry = exp(-logsOfHinverse[iH])
    Nx = Int(L÷ΔxTry) +1
    Δx = L/(Nx-1)
    X = [Δx * (i-1) for i ∈ range(1,Nx)]
    model=[Symbolics.value(substitute(β,Dict(x=>X[i]))) for i ∈ range(1,Nx)]
    lines!(ax, X, model)
end
display(fig)
Nx=nothing
Δx=nothing
iH=nothing
for iPointsUsed ∈ eachindex(numPointsX), iCase ∈ eachindex(cases), iH ∈ eachindex(logsOfHinverse)
    name,T,β = cases[iCase]
    iExperiment = (iH=iH,iCase=iCase,iPointsUsed=iPointsUsed)

    ΔxTry = exp(-logsOfHinverse[iH])
    Nx = Int(L÷ΔxTry) +1
    Δx = L/(Nx-1)
    X = [Δx * (i-1) for i ∈ range(1,Nx)]
    modelName = name*string(Nx)
    models=[]
    model=[Symbolics.value(substitute(β,Dict(x=>X[i]))) for i ∈ range(1,Nx)]
    models=push!(models, model)
    modelPoints = (Nx)
    tmpModel = (models=models, modelName=modelName, modelPoints=modelPoints,Δ=(Δx))
    modelFamily=push!(modelFamily,tmpModel)
    q = mySimplify(β*∂(T))
    qₓ = mySimplify(∂(q))
    force = [Symbolics.value(substitute(qₓ,Dict(x=>X[i]))) for i ∈ range(1,Nx)]
    

   
end


    

# input parameters

In [3]:

famousEquationType="1DpoissonHetero" #GT98
Δ = (1.0)

1.0

In [4]:


# orders: -1 -> indicator function, 0 -> box car, >=1 -> B-spline

orderBtime=1
orderBspace=1

pointsInSpace=3
pointsInTime=3

# the order of errors to be controlled
supplementaryOrder=2

# new parameters for interpolated Taylor expansion μ for field

fieldItpl = (ptsSpace = 1,ptsTime = 1,offsetSpace=1,offsetTime=1,YorderBspace=1,YorderBtime=1) #offsetSpace and offsetTime ∈ z 
# μ points should be distributed from y_min+offset*Δy to y_max-offset*Δy offset can be negative too


# new parameters for interpolated Taylor expansion μᶜ for material
materItpl = (ptsSpace = 1,ptsTime = 1,offsetSpace=1,offsetTime=1,YorderBspace=1,YorderBtime=1)



(ptsSpace = 1, ptsTime = 1, offsetSpace = 1, offsetTime = 1, YorderBspace = 1, YorderBtime = 1)

In [5]:
concreteParametersForOPTConstruction = @strdict famousEquationType Δ orderBtime orderBspace pointsInSpace pointsInTime supplementaryOrder fieldItpl materItpl


Dict{String, Any} with 9 entries:
  "fieldItpl"          => (ptsSpace = 1, ptsTime = 1, offsetSpace = 1, offsetTi…
  "Δ"                  => 1.0
  "supplementaryOrder" => 2
  "materItpl"          => (ptsSpace = 1, ptsTime = 1, offsetSpace = 1, offsetTi…
  "orderBspace"        => 1
  "orderBtime"         => 1
  "famousEquationType" => "1DpoissonHetero"
  "pointsInSpace"      => 3
  "pointsInTime"       => 3

In [6]:
optRec=myProduceOrLoad(makeOPTsemiSymbolic,concreteParametersForOPTConstruction,"semiSymbolic")

Dict{String, Any} with 2 entries:
  "gitcommit" => "c69609f9c80755006ae856b105881f4abe7a2a63-dirty"
  "recette"   => (lhs = (Ajiννᶜ = Num[0.5κ₁ + 0.5κ₂; -0.5κ₁ - κ₂ - 0.5κ₃; 0.5κ₂…

In [7]:
params=@strdict optRec=optRec modelFam=modelFamily[1] absorbingBoundaries=nothing maskedRegionInSpace=nothing

Dict{String, Any} with 4 entries:
  "absorbingBoundaries" => nothing
  "optRec"              => Dict{String, Any}("gitcommit"=>"c69609f9c80755006ae8…
  "maskedRegionInSpace" => nothing
  "modelFam"            => (models = Any[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  1, 1…

In [8]:
numOpt=numericalOperatorConstruction(params)

size(models[iVar]) = (32,)
newCoords = expandVectors(size(models[iVar]), CartesianDependency) = [32, 1]
ModelPoints[:, iVar] = newCoords = [32, 1]
tmpModel = reshape(models[iVar], newCoords...) = [1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1;;]
Models[iVar] = tmpModel = [1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1;;]
(Models, ModelPoints) = (Any[[1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1;;]], [32; 1;;])
size(models[iVar]) = (32,)
newCoords = expandVectors(size(models[iVar]), CartesianDependency) = [32, 1]
ModelPoints[:, iVar] = newCoords = [32, 1]
tmpModel = reshape(models[iVar], newCoords...) = [1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1;;]
Models[iVar] = tmpModel = [1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1; 1;;]
(Models, ModelPoint

Dict{String, @NamedTuple{costfunctions::Matrix{Num}, fieldLHS::Matrix{Any}, fieldRHS::Matrix{Any}, champsLimité::Matrix{Any}}} with 1 entry:
  "numOperators" => (costfunctions = [-2.0left_1_t_1₁ + left_1_t_1₂ - 0.834023r…

In [28]:

using JLD2

function prepareNumericalOperators(numOperators)
    @unpack costfunctions, fieldLHS, fieldRHS, champsLimité = numOperators

    costvec = vec(costfunctions)

    nA = ndims(fieldLHS)
    @assert nA >= 2 "fieldLHS must have at least [field, time] axes"

    timeAxis = nA
    fieldAxis = nA - 1
    spaceShape = size(fieldLHS)[1:fieldAxis-1]

    timePointsUsedForOneStep = size(fieldLHS, timeAxis)
    @show NField = size(fieldLHS, fieldAxis)

    Rcoord = CartesianIndices(spaceShape)
    linearR = LinearIndices(Rcoord)
    NpointsSpace = length(Rcoord)

    NknownTime = max(timePointsUsedForOneStep - 1, 0)

    symbUnknownField = Array{Num,2}(undef, NpointsSpace, NField)
    symbKnownField = Array{Num,3}(undef, NpointsSpace, NField, NknownTime)

    for iT in 1:NknownTime
        sliceT = selectdim(fieldLHS, timeAxis, iT)
        for iField in 1:NField
            A = selectdim(sliceT, fieldAxis, iField)
            for j in Rcoord
                lj = linearR[j]
                symbKnownField[lj, iField, iT] = A[j]
            end
        end
    end

    @show sliceT = selectdim(fieldLHS, timeAxis, timePointsUsedForOneStep)
    for iField in 1:NField
        A = selectdim(sliceT, fieldAxis, iField)
        for j in Rcoord
            lj = linearR[j]
            symbUnknownField[lj, iField] = A[j]
        end
    end

    if champsLimité === nothing
        symbKnownForce = Array{Num,3}(undef, NpointsSpace, NField, timePointsUsedForOneStep)
        for iT in 1:timePointsUsedForOneStep
            sliceT = selectdim(fieldRHS, timeAxis, iT)
            for iField in 1:NField
                A = selectdim(sliceT, fieldAxis, iField)
                for j in Rcoord
                    lj = linearR[j]
                    symbKnownForce[lj, iField, iT] = A[j]
                end
            end
        end
        NforcePoints = NpointsSpace
    else
        nC = ndims(champsLimité)
        @assert nC >= 2
        cTimeAxis = nC
        cFieldAxis = nC - 1
        NforcePoints = prod(size(champsLimité)[1:cFieldAxis-1])

        CRcoord = CartesianIndices(size(champsLimité)[1:cFieldAxis-1])
        ClinearR = LinearIndices(CRcoord)

        symbKnownForce = Array{Num,3}(undef, NforcePoints, NField, timePointsUsedForOneStep)
        for iT in 1:timePointsUsedForOneStep
            sliceT = selectdim(champsLimité, cTimeAxis, iT)
            for iField in 1:NField
                A = selectdim(sliceT, cFieldAxis, iField)
                for j in CRcoord
                    lj = ClinearR[j]
                    symbKnownForce[lj, iField, iT] = A[j]
                end
            end
        end
    end

    unknownInputs = vec(symbUnknownField)
    knownFieldInputs = NknownTime == 0 ? Num[] : vec(symbKnownField)
    knownForceInputs = vec(symbKnownForce)
    allInputs = vcat(unknownInputs, knownFieldInputs, knownForceInputs)

    residual_expr = build_function(costvec, allInputs; expression=Val(false))
    residual_fun = eval(residual_expr)

    unknown_template = zeros(Float64, NpointsSpace, NField)
    known_lhs_template = zeros(Float64, NpointsSpace, NField, NknownTime)
    known_rhs_template = zeros(Float64, NforcePoints, NField, timePointsUsedForOneStep)

    return (
        residual_fun = residual_fun,
        symbUnknownField = symbUnknownField,
        symbKnownField = symbKnownField,
        symbKnownForce = symbKnownForce,
        unknown_template = unknown_template,
        known_lhs_template = known_lhs_template,
        known_rhs_template = known_rhs_template,
        spaceShape = spaceShape,
        NpointsSpace = NpointsSpace,
        NforcePoints = NforcePoints,
        NField = NField,
        timePointsUsedForOneStep = timePointsUsedForOneStep,
        costvec = costvec,
    )
end


function makePreparedResidual(prepared)
    Nunknown = length(prepared.unknown_template)
    NknownLHS = length(prepared.known_lhs_template)
    NknownRHS = length(prepared.known_rhs_template)
    allInputs = zeros(Float64, Nunknown + NknownLHS + NknownRHS)

    function residual!(F, unknownField, knownField, knownForce)
        allInputs[1:Nunknown] .= vec(unknownField)
        allInputs[Nunknown+1:Nunknown+NknownLHS] .= vec(knownField)
        allInputs[Nunknown+NknownLHS+1:end] .= vec(knownForce)
        Ftmp = prepared.residual_fun(allInputs)
        F .= Ftmp
        return nothing
    end

    return residual!
end

function prepareSparseJacobian(prepared, residual!)
    unknownField = copy(prepared.unknown_template)
    knownField = copy(prepared.known_lhs_template)
    knownForce = copy(prepared.known_rhs_template)

    U = vec(unknownField)
    F = zeros(length(prepared.costvec))

    wrapped! = (F, U) -> begin
        unknownField .= reshape(U, size(unknownField))
        residual!(F, unknownField, knownField, knownForce)
    end

    sparsity = Symbolics.jacobian_sparsity(wrapped!, F, U)
    J = Float64.(sparse(sparsity))
    cache = ForwardColorJacCache(wrapped!, rand(length(U)))
    return J, cache
end

function timeMarchingSchemePrepared(prepared, Nt, Δnum, modelName;
    videoMode=true,
    sourceType="Ricker",
    t₀=50,
    f₀=0.04,
    initialCondition=0.0,
    sourceFull=nothing,
    iExperiment=nothing,
)

    if !isdir(datadir("fieldResults"))
        mkdir(datadir("fieldResults"))
    end

    @unpack iH, iCase, iPointsUsed = iExperiment
    experiment_name = "$(iH)_$(iCase)_$(iPointsUsed)"
    sequentialFileName = datadir("fieldResults", savename((Nt, Δnum..., modelName, sourceType, experiment_name), "jld2"))
    compactFileName = datadir("fieldResults", savename("compact", (Nt, Δnum..., modelName, sourceType, experiment_name), "jld2"))

    residual! = makePreparedResidual(prepared)
    J, cache = prepareSparseJacobian(prepared, residual!)

    NField = prepared.NField
    NpointsSpace = prepared.NpointsSpace
    timePointsUsedForOneStep = prepared.timePointsUsedForOneStep
    spaceShape = prepared.spaceShape
    NknownTime = max(timePointsUsedForOneStep - 1, 0)

    unknownField = fill(initialCondition, size(prepared.unknown_template))
    knownField = fill(initialCondition, size(prepared.known_lhs_template))
    knownForce = fill(initialCondition, size(prepared.known_rhs_template))

    itVec = collect(1:Nt)
    t = (itVec .- 1) .* Δnum[end]

    sourceTime = nothing
    if sourceType == "Ricker"
        myRicker(x) = Ricker(x, t₀, f₀)
        sourceTime = myRicker.(t)
        if timePointsUsedForOneStep != 1
            prepend!(sourceTime, zeros(timePointsUsedForOneStep))
        end
    elseif sourceType == "Explicit"
        expected = (prepared.NforcePoints, NField, Nt)
        size(sourceFull) == expected || error("sourceFull should have size $expected")
    end

    F = zeros(length(prepared.costvec))

    if !isfile(sequentialFileName)
        fieldFile = jldopen(sequentialFileName, "w")

        if videoMode
            fig = Figure()
            ax = Axis(fig[1, 1])
            firstField = reshape(unknownField[:, 1], spaceShape...)
            hm = heatmap!(ax, Float32.(firstField), colormap=:deep, colorrange=(-1e-5, 1e-5))
            display(fig)
        end

        for it in itVec
            if sourceType == "Ricker"
                knownForce .= reshape(sourceTime[it:it+timePointsUsedForOneStep-1], 1, 1, :)
            else
                knownForce .= sourceFull[:, :, it:it+timePointsUsedForOneStep-1]
            end

            timeStepOptimisationPrepared!(residual!, unknownField, knownField, knownForce, J, cache, F)

            if NknownTime > 0
                if NknownTime > 1
                    knownField[:, :, 1:end-1] .= knownField[:, :, 2:end]
                end
                knownField[:, :, end] .= unknownField
            end

            newField = reshape(unknownField, NpointsSpace, NField)
            fieldFile["timestep_$it"] = reshape(newField, spaceShape..., NField)

            if videoMode
                firstField = reshape(unknownField[:, 1], spaceShape...)
                hm[1] = Float32.(firstField)
            end
        end

        close(fieldFile)
    end

    file = jldopen(sequentialFileName, "r")
    a = [file["timestep_$it"] for it in itVec]
    close(file)

    if videoMode
        fig = Figure()
        ax = Axis(fig[1, 1])
        hm = heatmap!(ax, Float32.(selectdim(a[1], ndims(a[1]), 1)), colormap=:plasma, colorrange=(-1e-6, 1e-6))
        Colorbar(fig[1, 2], hm)
        display(fig)
    end

    acompact = reduce(hcat, a)
    @save compactFileName acompact compress=true

    return acompact
end

function timeStepOptimisationPrepared!(residual!, unknownField, knownField, knownForce, J, cache, F;
    nIteration=10, smallNumber=1e-8, boundaryConditionForced=false)

    nEq = length(F)
    normalisation = 1.0 / nEq
    r1 = 1.0

    U = vec(copy(unknownField))

    wrapped! = (F, U) -> begin
        unknownField .= reshape(U, size(unknownField))
        residual!(F, unknownField, knownField, knownForce)
        if boundaryConditionForced
            F[1] = U[1] - 1.0
            F[end] = U[end] - 1.0
        end
        nothing
    end

    for iter in 1:nIteration
        wrapped!(F, U)

        r = norm(F) * normalisation
        if iter == 1
            r1 = r
        end
        if r == 0.0 || r / r1 < smallNumber
            break
        end

        forwarddiff_color_jacobian!(J, wrapped!, U, cache)
        δU = -(J \ F)
        U .+= δU
    end

    unknownField .= reshape(U, size(unknownField))
    return nothing
end


timeStepOptimisationPrepared! (generic function with 1 method)

In [29]:
@unpack costfunctions, fieldLHS, fieldRHS, champsLimité = numOps
fieldLHS

1×1 Matrix{Any}:
 Num[left_1_t_1₁, left_1_t_1₂, left_1_t_1₃, left_1_t_1₄, left_1_t_1₅, left_1_t_1₆, left_1_t_1₇, left_1_t_1₈, left_1_t_1₉, left_1_t_1₁₀  …  left_1_t_1₂₃, left_1_t_1₂₄, left_1_t_1₂₅, left_1_t_1₂₆, left_1_t_1₂₇, left_1_t_1₂₈, left_1_t_1₂₉, left_1_t_1₃₀, left_1_t_1₃₁, left_1_t_1₃₂]

In [30]:
numOps = numOpt["numOperators"]
prepared = prepareNumericalOperators(numOps)

syntheticData = timeMarchingSchemePrepared(
    prepared,
    Nt,
    Δnum,
    modelName;
    videoMode=false,
    sourceType="Explicit",
    sourceFull=sourceFull,
    iExperiment=iExperiment,
)


NField = size(fieldLHS, fieldAxis) = 1
sliceT = selectdim(fieldLHS, timeAxis, timePointsUsedForOneStep) = Any[Num[left_1_t_1₁, left_1_t_1₂, left_1_t_1₃, left_1_t_1₄, left_1_t_1₅, left_1_t_1₆, left_1_t_1₇, left_1_t_1₈, left_1_t_1₉, left_1_t_1₁₀, left_1_t_1₁₁, left_1_t_1₁₂, left_1_t_1₁₃, left_1_t_1₁₄, left_1_t_1₁₅, left_1_t_1₁₆, left_1_t_1₁₇, left_1_t_1₁₈, left_1_t_1₁₉, left_1_t_1₂₀, left_1_t_1₂₁, left_1_t_1₂₂, left_1_t_1₂₃, left_1_t_1₂₄, left_1_t_1₂₅, left_1_t_1₂₆, left_1_t_1₂₇, left_1_t_1₂₈, left_1_t_1₂₉, left_1_t_1₃₀, left_1_t_1₃₁, left_1_t_1₃₂]]


MethodError: MethodError: Cannot `convert` an object of type Vector{Num} to an object of type Num
The function `convert` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  Num(::Any)
   @ Symbolics ~/.julia/packages/Symbolics/xD5Pj/src/num.jl:2
  convert(::Type{Num}, !Matched::Num)
   @ Symbolics ~/.julia/packages/Symbolics/xD5Pj/src/num.jl:256
  convert(::Type{Num}, !Matched::SymbolicUtils.Symbolic{<:Number})
   @ Symbolics ~/.julia/packages/Symbolics/xD5Pj/src/num.jl:254
  ...


In [10]:
Nt = 1
            
            
syntheticData=timeMarchingScheme(numOpt, Nt, Δnum,modelName;videoMode=false,sourceType="Explicit",sourceFull=sourceFull,iExperiment=iExperiment)
syntheticData=reduce(vcat,syntheticData)
analyticalData = [Symbolics.value(substitute(u,Dict(x=>X[i]))) for i ∈ range(1,Nx)]

UndefVarError: UndefVarError: `sourceFull` not defined in `Main`
Suggestion: check for spelling errors or missing imports.